In [ ]:
%pip install langchain langgraph openai typing

In [ ]:
from typing import TypedDict, Annotated
import operator

class llmSettings(TypedDict, total=False):
  prompt: str
  api_key: str
  llm_publisher: str
  llm_model: str
  target_text_length: int
  target_channel: str


class GraphState(TypedDict, total=False):
  keyword: str
  need_keyword: bool
  keywords: list[str]
  settings: llmSettings
  products: Annotated[dict[str, list[dict]], operator.or_]
  filtered_products: list[dict]
  need_retry: bool



In [ ]:
import random
import asyncio

queue_size = 5

# 시작 노드
async def entry_node(state: GraphState) -> GraphState:
  # 키워드가 없을 경우
  # 키워드 필요함!
  if state.get("keyword", "") is None:
    return {"need_keyword": True}
  # 키워드가 있을 경우
  # 키워드 안 필요함!
  return {
    "keywords": [state.get("keyword")],
    "need_keyword": False
  }

# 키워드 없어서 가져오는 노드
async def crawling_keywords_node(state: GraphState) -> GraphState:
  # state.get("target_channel", "")에
    # x가 포함되어있다면 x에서 키워드 가져오기
    # i가 포함되어있다면 instagram에서 해시태그 가져오기
    # g가 포함되어있다면 google trend에서 크롤링
  
  # TODO: 예시 결과입니다. 실제 로직으로 수정 필요
  keywords = ["평택대", "bangladesh vs ireland", "나경원", "중앙대학교", "강백호", "메이플", "조달청", "국립중앙박물관", "마이애미 대 골든 스테이트", "한국장학재단"]
  return {"keywords": keywords}

async def make_keyword_node(state: GraphState) -> GraphState:
  # 배열을 주고, 해당 배열 중 하나를 선택하고, 출력물로 하나의 품목을 검색하기 위한 키워드를 뱉음
  # LLM이 키워드를 정할 예정
  keyword = "캐릭터 볼펜"
  return {"keyword": keyword}

async def get_keyword_node(state: GraphState) -> GraphState:
  keywords = state.get("keywords", [""])

  # TODO: 랜덤으로 하나 뽑기. 나중에 수정하고싶으면 상의하세요
  keyword = keywords[random.randrange(0, len(keywords))]
  return {"keyword": keyword}

async def crawling_items_ssadagu_node(state: GraphState) -> GraphState:
  # 싸다구 몰에서 아이템 크롤링
  # TODO: 예시 결과입니다. 실제 로직으로 수정 필요
  result = [
    {
      "title": "2025 새로운 국경 스마트 폰 I16PROMax 안드로이드 전화 AliExpress 핫 세일 새로운 공장 도매",
      "price": "41800",
      "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=901876889270&ss_tx=스마트폰",
      "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01Kkep2t2MGSUYrhH3X_!!2217178229800-0-cib.jpg",
      "sales_count": "0"
    },
  ]
  return {**state.get("products", {}), "ssadagu":result}

async def crawling_items_coupang_node(state: GraphState) -> GraphState:
  # 쿠팡에서 크롤링
  # TODO: 예시 결과입니다. 실제 로직으로 수정 필요
    # 크롤링 테스트가 나오면 예시도 수정이 필요합니다.
  result = [
    {
      "title": "2025 새로운 국경 스마트 폰 I16PROMax 안드로이드 전화 AliExpress 핫 세일 새로운 공장 도매",
      "price": "41800",
      "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=901876889270&ss_tx=스마트폰",
      "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01Kkep2t2MGSUYrhH3X_!!2217178229800-0-cib.jpg",
      "sales_count": "0"
    },
  ]
  return {**state.get("products", {}), "coupang":result}
  
async def filter_strange_node(state: GraphState) -> GraphState:
  products = state["products"]
  keyword = state["keyword"]

  semaphore = asyncio.Semaphore(5)

  async def filter_strange(product, keyword) -> dict:
    async with semaphore:
      # TODO: LLM에게 질문, 해당 제품이 키워드와 연관이 충분히 있는가?
        # 없으면 빈 칸 출력
        # 있으면 그대로 출력
      return product

  # TODO: 이렇게 하면 products["ssadagu"]와 products["coupang"] 내부의 아이템을 직접 검사하기는 어려울거임.. 어떻게 바꿔야 하지
    """
    결과값은 다음과 같이 생성되어야 함:
    {
      "ssadagu": [
        {
          "title": "2025 새로운 국경 스마트 폰 I16PROMax 안드로이드 전화 AliExpress 핫 세일 새로운 공장 도매",
          "price": "41800",
          "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=901876889270&ss_tx=스마트폰",
          "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01Kkep2t2MGSUYrhH3X_!!2217178229800-0-cib.jpg",
          "sales_count": "0"
        },
        ...
      ],
      "coupang": [
        {
          "title": "2025 새로운 국경 스마트 폰 I16PROMax 안드로이드 전화 AliExpress 핫 세일 새로운 공장 도매",
          "price": "41800",
          "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=901876889270&ss_tx=스마트폰",
          "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01Kkep2t2MGSUYrhH3X_!!2217178229800-0-cib.jpg",
          "sales_count": "0"
        },
        ...
      ],
      ...
    }

    이때, 각 배열이 비었을 경우 각 속성 자체가 없어져야 함
    그런데 지금 코드에서 결과값은 하나의 dict 배열로만 만들어질 텐데, 어떻게 이걸 구현하지?
    """
  results = await asyncio.gather(*(filter_strange(product) for product in products))

  return results

async def product_check(state: GraphState) -> GraphState:
  malls = state.get("filtered_products")
  
  # 만약 malls가 2개 미만이라면? 다시 돌기
  if len(malls) < 2:
    return {"need_retry": True}
  
  # TODO: 각 검색한 상품들이 유사한가?
    # 유사하지 않다면 다시 돌기
    # 유사하면 그냥 넘기기
  return {"need_retry": True}

async def generate_ads(state: GraphState) -> GraphState:
  # TODO: 설정 갖고 각 플랫폼의 성격에 맞게 LLM이 글 쓰기
  # 이건 그냥 if 문으로 순회해도 될 듯? 어짜피 값을 요구하는게 아니라 로직 돌고 있다고 나중에 통보만 할거라
  content1={
    "title": "블로그 제목",
    "content": "블로그 내용",
    "tags": ["#해시", "#태그들"]
  }
  content2={
    "content": "트윗 내용",
    "images": ["이미지 링크. 없으면 빈칸"]
  }
  content3={
    "content": "쓰레드 내용",
    "images": ["이미지 링크. 없으면 빈칸"]
  }

  # 실제로는 이렇게 단순하게 보내진 않습니다. 예시 출력이 다음과 같다 이 말입니다.
  return {
    "blog": content1,
    "x": content2,
    "thread": content3
  }


_IncompleteInputError: incomplete input (961075424.py, line 90)

In [ ]:
from langgraph.graph import StateGraph

workflow = StateGraph(GraphState)

workflow.add_node("entry_node", entry_node)
workflow.add_node("crawling_keywords_node", crawling_keywords_node)
workflow.add_node("make_keyword_node", make_keyword_node)
workflow.add_node("get_keyword_node", get_keyword_node)
workflow.add_node("crawling_items_ssadagu_node", crawling_items_ssadagu_node)
workflow.add_node("crawling_items_coupang_node", crawling_items_coupang_node)
# 조인 노드 필요
workflow.add_node("filter_strange_node", filter_strange_node)
workflow.add_node("product_check", product_check)
workflow.add_node("generate_ads", generate_ads)

# TODO: entry_node를 거친 후 state에 keyword가 있다면 get_keyword_node
  # 없으면 crawling_keywords_node로 가게 하기
workflow.add_conditional_edges(
  "entry_node",
)

workflow.add_edge("crawling_keywords_node", "make_keyword_node")
workflow.add_node("make_keyword_node", "get_keyword_node")

workflow.add_edge("get_keyword_node", "crawling_items_ssadagu_node")
workflow.add_edge("get_keyword_node", "crawling_items_coupang_node")
# TODO: 이 결과물을 어떻게 기다리지?
  # 모두 도착했을 때 product_check로 들어가게 하고 싶은데.. 

workflow.add_edge("product_check", "generate_ads")

workflow.set_entry_point("entry_node")
workflow.set_finish_point("generate_ads")

In [ ]:
app = workflow.compile()

In [ ]:
# 키워드가 주어졌을 경우
state1 = GraphState(
  keyword="남성 티셔츠"
)

# 안 주어졌을 경우
state2 = GraphState()

In [ ]:
app.ainvoke(state1)

In [ ]:
app.ainvoke(state2)